### Search Engine With Tools and Agents

In [1]:
## Arxiv -- Research 
## Tools Creation 
from langchain_community.tools import ArxivQueryRun, WikipediaQueryRun
from langchain_community.utilities import WikipediaAPIWrapper,ArxivAPIWrapper

In [2]:
## use the inbuilt tool of wikipedia 

api_wrapper_wiki=WikipediaAPIWrapper(top_k_results=1,doc_content_chars_max=250)
wiki=WikipediaQueryRun(api_wrapper=api_wrapper_wiki)
wiki.name


'wikipedia'

In [3]:
api_wrapper_arxiv=ArxivAPIWrapper(top_k_results=1,doc_content_chars_max=250)
arxiv=ArxivQueryRun(api_wrapper=api_wrapper_arxiv)
print(arxiv.name)

arxiv


In [4]:
tools=[wiki,arxiv]

In [6]:
## Custom tools[RAG Tool]
from langchain_community.document_loaders import WebBaseLoader
from langchain_community.vectorstores import FAISS
from langchain_huggingface import HuggingFaceEmbeddings
from langchain_text_splitters import RecursiveCharacterTextSplitter

USER_AGENT environment variable not set, consider setting it to identify your requests.


In [7]:
embeddings=HuggingFaceEmbeddings(model_name="all-MiniLM-L6-v2")

In [8]:
loader=WebBaseLoader("https://docs.smith.langchain.com/")
docs=loader.load()
documents=RecursiveCharacterTextSplitter(chunk_size=1000,chunk_overlap=200).split_documents(docs)
vectordb=FAISS.from_documents(documents,embeddings)
retriever=vectordb.as_retriever()
retriever

VectorStoreRetriever(tags=['FAISS', 'HuggingFaceEmbeddings'], vectorstore=<langchain_community.vectorstores.faiss.FAISS object at 0x00000269A707A660>, search_kwargs={})

In [11]:
from langchain_core.tools.retriever import create_retriever_tool
retriever_tool=create_retriever_tool(retriever,"langsmith-search","Search any information about Langsmith ")

retriever_tool.name


'langsmith-search'

In [12]:
tools=[wiki,arxiv,retriever_tool]

In [13]:
tools

[WikipediaQueryRun(api_wrapper=WikipediaAPIWrapper(wiki_client=<module 'wikipedia' from 'e:\\Langchain_projects\\venv\\Lib\\site-packages\\wikipedia\\__init__.py'>, top_k_results=1, lang='en', load_all_available_meta=False, doc_content_chars_max=250)),
 ArxivQueryRun(api_wrapper=ArxivAPIWrapper(arxiv_search=<class 'arxiv.Search'>, arxiv_exceptions=(<class 'arxiv.ArxivError'>, <class 'arxiv.UnexpectedEmptyPageError'>, <class 'arxiv.HTTPError'>), top_k_results=1, ARXIV_MAX_QUERY_LENGTH=300, continue_on_failure=False, load_max_docs=100, load_all_available_meta=False, doc_content_chars_max=250)),
 StructuredTool(name='langsmith-search', description='Search any information about Langsmith ', args_schema=<class 'langchain_core.tools.retriever.RetrieverInput'>, func=<function create_retriever_tool.<locals>.func at 0x00000269C4BF3C40>, coroutine=<function create_retriever_tool.<locals>.afunc at 0x00000269819DFA60>)]

In [14]:
## Run all this tools agents and llm models 

## Tools, LLM-->AgentExecutor
from langchain_groq import ChatGroq
from dotenv import load_dotenv

load_dotenv()
import os

groq_api_key=os.getenv("GROQ_API_KEY")


llm=ChatGroq(groq_api_key=groq_api_key,model_name="qwen/qwen3-32b")

In [42]:
from langchain_classic import hub # ✅ Correct import

In [15]:
## Prompt Template
from langchain_classic import hub
prompt=hub.pull("hwchase17/openai-functions-agent")
prompt.messages

[SystemMessagePromptTemplate(prompt=PromptTemplate(input_variables=[], input_types={}, partial_variables={}, template='You are a helpful assistant'), additional_kwargs={}),
 MessagesPlaceholder(variable_name='chat_history', optional=True),
 HumanMessagePromptTemplate(prompt=PromptTemplate(input_variables=['input'], input_types={}, partial_variables={}, template='{input}'), additional_kwargs={}),
 MessagesPlaceholder(variable_name='agent_scratchpad')]

In [16]:
## Agents
from langchain_classic.agents import create_openai_tools_agent
agent=create_openai_tools_agent(llm,tools,prompt)
agent

RunnableAssign(mapper={
  agent_scratchpad: RunnableLambda(lambda x: format_to_openai_tool_messages(x['intermediate_steps']))
})
| ChatPromptTemplate(input_variables=['agent_scratchpad', 'input'], optional_variables=['chat_history'], input_types={'chat_history': list[typing.Annotated[typing.Union[typing.Annotated[langchain_core.messages.ai.AIMessage, Tag(tag='ai')], typing.Annotated[langchain_core.messages.human.HumanMessage, Tag(tag='human')], typing.Annotated[langchain_core.messages.chat.ChatMessage, Tag(tag='chat')], typing.Annotated[langchain_core.messages.system.SystemMessage, Tag(tag='system')], typing.Annotated[langchain_core.messages.function.FunctionMessage, Tag(tag='function')], typing.Annotated[langchain_core.messages.tool.ToolMessage, Tag(tag='tool')], typing.Annotated[langchain_core.messages.ai.AIMessageChunk, Tag(tag='AIMessageChunk')], typing.Annotated[langchain_core.messages.human.HumanMessageChunk, Tag(tag='HumanMessageChunk')], typing.Annotated[langchain_core.messages

In [17]:
## Agent Executer
from langchain_classic.agents import AgentExecutor
agent_executor=AgentExecutor(agent=agent,tools=tools,verbose=True)
agent_executor

AgentExecutor(verbose=True, agent=RunnableMultiActionAgent(runnable=RunnableAssign(mapper={
  agent_scratchpad: RunnableLambda(lambda x: format_to_openai_tool_messages(x['intermediate_steps']))
})
| ChatPromptTemplate(input_variables=['agent_scratchpad', 'input'], optional_variables=['chat_history'], input_types={'chat_history': list[typing.Annotated[typing.Union[typing.Annotated[langchain_core.messages.ai.AIMessage, Tag(tag='ai')], typing.Annotated[langchain_core.messages.human.HumanMessage, Tag(tag='human')], typing.Annotated[langchain_core.messages.chat.ChatMessage, Tag(tag='chat')], typing.Annotated[langchain_core.messages.system.SystemMessage, Tag(tag='system')], typing.Annotated[langchain_core.messages.function.FunctionMessage, Tag(tag='function')], typing.Annotated[langchain_core.messages.tool.ToolMessage, Tag(tag='tool')], typing.Annotated[langchain_core.messages.ai.AIMessageChunk, Tag(tag='AIMessageChunk')], typing.Annotated[langchain_core.messages.human.HumanMessageChunk, Tag

In [18]:
agent_executor.invoke({"input":"Tell me about Langsmith"})



> Entering new AgentExecutor chain...

Invoking: `langsmith-search` with `{'query': 'Langsmith'}`


LangSmith docs - Docs by LangChainSkip to main contentDocs by LangChain home pageLangSmithSearch...⌘KAsk AIGitHubTry LangSmithTry LangSmithSearch...NavigationLangSmith docsGet startedObservabilityEvaluationPrompt engineeringDeploymentPlatform setupReferenceOverviewCreate an account and API keyIntegrationsPlansAccount administrationOverviewSet up hierarchyWorkload isolationManage organizations using the APIManage billingGranular usageSet up resource tagsUser managementAdditional resourcesPolly AI assistantBetaData managementAccess control & AuthenticationScalability & resilienceFAQsRegions FAQPricing FAQLangSmith statusLangSmith docsCopy pageCopy pageLangSmith is a framework-agnostic platform for developing, debugging, and deploying AI agents and LLM applications.

It helps you trace requests, evaluate outputs, test prompts, and manage deployments in one place. LangSmith works with any 

{'input': 'Tell me about Langsmith',
 'output': "LangSmith is a **framework-agnostic platform** designed for building, debugging, and deploying AI agents and large language model (LLM) applications. It provides an integrated workflow for developers, offering tools for **observability, evaluation, prompt engineering, and deployment** all in one place. Here's a breakdown of its key features and capabilities:\n\n---\n\n### **Core Features**\n1. **Observability**  \n   - Trace requests and debug applications by tracking every step of the agent’s execution.\n   - Gain visibility into inputs, outputs, and intermediate steps for reliability and performance analysis.\n\n2. **Evaluation**  \n   - Measure and track the quality of AI outputs over time using customizable metrics.\n   - Validate consistency, accuracy, and trustworthiness of models or agents.\n\n3. **Prompt Engineering**  \n   - Test and iterate on prompts with versioning, collaboration tools, and built-in feedback loops.\n   - Opti

In [19]:
agent_executor.invoke({"input":"What is machine learning"})



> Entering new AgentExecutor chain...

Invoking: `wikipedia` with `{'query': 'machine learning'}`


Page: Machine learning
Summary: Machine learning (ML) is a field of study in artificial intelligence concerned with the development and study of statistical algorithms that can learn from data and generalize to unseen data, and thus perform tasks witMachine learning (ML) is a branch of artificial intelligence (AI) that focuses on developing algorithms and models capable of learning patterns and making decisions from data. Instead of relying on explicit programming instructions, ML systems improve their performance over time by analyzing examples and adjusting their parameters accordingly.

**Key points:**
- **Learning from data**: ML algorithms identify patterns in historical or real-time data to make predictions or decisions.
- **Generalization**: They aim to apply learned knowledge to new, unseen data (e.g., classifying images, recommending products).
- **Applications**: Used in area

{'input': 'What is machine learning',
 'output': 'Machine learning (ML) is a branch of artificial intelligence (AI) that focuses on developing algorithms and models capable of learning patterns and making decisions from data. Instead of relying on explicit programming instructions, ML systems improve their performance over time by analyzing examples and adjusting their parameters accordingly.\n\n**Key points:**\n- **Learning from data**: ML algorithms identify patterns in historical or real-time data to make predictions or decisions.\n- **Generalization**: They aim to apply learned knowledge to new, unseen data (e.g., classifying images, recommending products).\n- **Applications**: Used in areas like speech recognition, medical diagnosis, self-driving cars, and more.\n- **Types**: Common approaches include supervised learning (labeled data), unsupervised learning (unlabeled data), and reinforcement learning (learning through trial/error with rewards).\n\nAt its core, machine learning e

In [20]:
agent_executor.invoke({"input":"What's the paper 1706.03762 about?"})



> Entering new AgentExecutor chain...

Invoking: `arxiv` with `{'query': '1706.03762'}`


Published: 2023-08-02
Title: Attention Is All You Need
Authors: Ashish Vaswani, Noam Shazeer, Niki Parmar, Jakob Uszkoreit, Llion Jones, Aidan N. Gomez, Lukasz Kaiser, Illia Polosukhin
Summary: The dominant sequence transduction models are based on cThe paper with ID [1706.03762](https://arxiv.org/abs/1706.03762) is titled **"Attention Is All You Need"** and is one of the most influential works in modern natural language processing (NLP). Here's a summary of its key contributions:

### Key Points:
- **Transformers**: Introduces the **Transformer architecture**, which replaces traditional sequence modeling approaches (like RNNs/LSTMs) with a purely attention-based design.
- **Self-Attention Mechanism**: Proposes the **self-attention mechanism**, allowing models to dynamically weigh the importance of different parts of an input sequence relative to each other.
- **Parallelization**: Eliminates seq

{'input': "What's the paper 1706.03762 about?",
 'output': 'The paper with ID [1706.03762](https://arxiv.org/abs/1706.03762) is titled **"Attention Is All You Need"** and is one of the most influential works in modern natural language processing (NLP). Here\'s a summary of its key contributions:\n\n### Key Points:\n- **Transformers**: Introduces the **Transformer architecture**, which replaces traditional sequence modeling approaches (like RNNs/LSTMs) with a purely attention-based design.\n- **Self-Attention Mechanism**: Proposes the **self-attention mechanism**, allowing models to dynamically weigh the importance of different parts of an input sequence relative to each other.\n- **Parallelization**: Eliminates sequential dependencies, enabling faster training via parallel computation.\n- **Impact**: Laid the foundation for models like BERT, GPT, and other state-of-the-art NLP systems.\n\n### Authors:\nAshish Vaswani, Noam Shazeer, Niki Parmar, Jakob Uszkoreit, Llion Jones, Aidan N. Go

Failed to refresh cache entry hwchase17/openai-functions-agent: Connection error caused failure to GET /commits/hwchase17/openai-functions-agent/latest in LangSmith API. Please confirm your internet connection. ConnectionError(MaxRetryError('HTTPSConnectionPool(host=\'api.smith.langchain.com\', port=443): Max retries exceeded with url: /commits/hwchase17/openai-functions-agent/latest (Caused by NameResolutionError("HTTPSConnection(host=\'api.smith.langchain.com\', port=443): Failed to resolve \'api.smith.langchain.com\' ([Errno 11001] getaddrinfo failed)"))'))
Content-Length: None
API Key: lsv2_********************************************99
Failed to refresh cache entry hwchase17/openai-functions-agent: Connection error caused failure to GET /commits/hwchase17/openai-functions-agent/latest in LangSmith API. Please confirm your internet connection. ConnectionError(MaxRetryError('HTTPSConnectionPool(host=\'api.smith.langchain.com\', port=443): Max retries exceeded with url: /commits/hwch

In [22]:
from langchain_classic.agents import initialize_agent,AgentType

In [24]:
from langchain_classic.callbacks import StreamlitCallbackHandler

In [25]:
!pip install -U ddgs

   ---------------------------------------- 0.0/4.0 MB ? eta -:--:--
   -- ------------------------------------- 0.3/4.0 MB ? eta -:--:--
   ----- ---------------------------------- 0.5/4.0 MB 2.0 MB/s eta 0:00:02
   ---------- ----------------------------- 1.0/4.0 MB 1.9 MB/s eta 0:00:02
   --------------- ------------------------ 1.6/4.0 MB 2.1 MB/s eta 0:00:02
   ----------------------- ---------------- 2.4/4.0 MB 2.3 MB/s eta 0:00:01
   ---------------------------- ----------- 2.9/4.0 MB 2.4 MB/s eta 0:00:01
   --------------------------------- ------ 3.4/4.0 MB 2.4 MB/s eta 0:00:01
   ------------------------------------ --- 3.7/4.0 MB 2.4 MB/s eta 0:00:01
   ---------------------------------------- 4.0/4.0 MB 2.3 MB/s  0:00:01
   ---------------------------------------- 0.0/3.5 MB ? eta -:--:--
   -- ------------------------------------- 0.3/3.5 MB ? eta -:--:--
   ----------- ---------------------------- 1.0/3.5 MB 2.7 MB/s eta 0:00:01
   ----------------- ----------------------